In [91]:
from dotenv import load_dotenv
load_dotenv()

True

In [92]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool

@tool
def web_search(query: str)-> str:
    "Search the web for recipes and cooking ideas."
    search_client = DuckDuckGoSearchRun()
    result = search_client.invoke(query)
    return result

answer = web_search.invoke("Banana Recipes.")

In [99]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

gemini_model = ChatGoogleGenerativeAI(model = "gemini-2.5-flash", max_retries = 2, temperature = 0.8)
model_with_tools = gemini_model.bind_tools([web_search])
system_prompt = """You are an Expert Chef who can cook anything from given ingredients.

Your job is to suggest creative and delicious recipes.
use the web_search tool to find inspiration.
Always respond helpfully.
"""
agent = create_agent(
    model = model_with_tools,
    tools= [web_search],
    system_prompt= system_prompt,
)

In [94]:
config = {"configurable":{"thread_id": "1"}}

inputs = {
    "messages": [("human", "what can I cook with guvar fali?")]
}

for chunk in agent.stream(inputs, stream_mode="values"):
    print(chunk["messages"][-1].pretty_print())

================================ Human Message =================================

what can I cook with guvar fali?
None
================================== Ai Message ==================================
Tool Calls:
  web_search (e16e6066-6d61-4d54-aa82-3a5ebc42fc93)
 Call ID: e16e6066-6d61-4d54-aa82-3a5ebc42fc93
  Args:
    query: guvar fali recipes
None
================================= Tool Message =================================
Name: web_search

Dahi varu Guvar nu Shak / Curded Cluster Beans / Gavar Fali in Curd ... Categories: Healthy recipes ... Subscribe to explore various recipes... Next Article Next Article: દહી વારુ ગુવાર નું શાક / Dahi varu Guvar nu Shak / Curded Cluster Beans / Gavar Fali ... ... Article Previous Article: દહી વારુ ગુવાર નું શાક / Dahi varu Guvar nu Shak / Curded Cluster Beans / Gavar Fali in ... Vegetables name in hindi – Guar, Gawar fali, Guwar ... Gavar, Guvar (Gujrati) other cuisines include Ker-Sangr, Jodhpuri Gatta, Tarfani, Raab di, Panchkoota, Chaava

In [ ]:
#TODO Connect it with Aditi Microwave and Refrigerator.

In [95]:
from IPython.display import display
from ipywidgets import FileUpload

uploader = FileUpload(accept = "image/*" , multiple = False)
display(uploader)

FileUpload(value=(), accept='image/*', description='Upload')

In [96]:
print(uploader)
print(uploader.value[0].content)

# memory view 
mv = uploader.value[0].content

byte_img = bytes(mv)
print(byte_img)
import base64
img_b64 = base64.b64encode(byte_img).decode("utf-8")


FileUpload(value=({'name': 'open_fridge.jpg', 'type': 'image/jpeg', 'size': 165371, 'content': <memory at 0x0000028396A59300>, 'last_modified': datetime.datetime(2026, 5, 26, 15, 10, 21, 414000, tzinfo=datetime.timezone.utc)},), accept='image/*', description='Upload')
b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x01,\x01,\x00\x00\xff\xdb\x00C\x00\n\x07\x07\x08\x07\x06\n\x08\x08\x08\x0b\n\n\x0b\x0e\x18\x10\x0e\r\r\x0e\x1d\x15\x16\x11\x18#\x1f%$"\x1f"!&+7/&)4)!"0A149;>>>%.DIC<H7=>;\xff\xdb\x00C\x01\n\x0b\x0b\x0e\r\x0e\x1c\x10\x10\x1c;("(;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;\xff\xc0\x00\x11\x08\x05n\x03L\x03\x01\x11\x00\x02\x11\x01\x03\x11\x01\xff\xc4\x00\x1c\x00\x00\x02\x03\x01\x01\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x02\x03\x04\x05\x06\x07\x08\xff\xc4\x00b\x10\x00\x01\x03\x02\x04\x02\x05\x05\x0b\x06\n\x07\x03\x0b\x02\x07\x01\x00\x02\x03\x04\x11\x05\x12!1AQ\x06\x13"aq\x142R\x81\x91\x15\x16#BST\x92\xa1\xb1\xc1\xd1\x07\x173br\x93$4CDV\x82\x94\xa4\xe1\xe35Uc\x

In [97]:
input = HumanMessage(
    content = [
        {
            "type" : "text",
            "text" : "Based on the ingredients you can see in the image, suggest me some recipes."
        },
        {
            "type" : "image",
            "base64" : img_b64,
            "mime_type" : "image/jpeg"
        }
    ]
)

In [100]:
for chunk in agent.stream({"messages": [input]}, stream_mode="values"):
    print(chunk["messages"][-1].pretty_print())

================================ Human Message =================================

[{'type': 'text', 'text': 'Based on the ingredients you can see in the image, suggest me some recipes.'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/jpeg'}]
None


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}